# V3 graph diagnostics — merytoryczna eksploracja przed modelowaniem

Notebook łączy diagnostykę struktury DAG z analizą gotowości do GNN.
Analizujemy głównie **strukturę grafu** (`nodes` + `edges`); scenariusz `clean`
używamy tylko do porównania rankingu leków z syntetycznymi proxy ADR.

Zakres:

1. Warstwy, typy węzłów i integralność DAG
2. Stopnie, huby, źródła, ujścia, izolaty
3. Głębokość topologiczna (warstwy przyczynowe)
4. Typy krawędzi i clinical pathways
5. Ścieżki do endpointów
6. Motywy przyczynowe: mediacja, fork, collider
7. Przodkowie i potomkowie endpointów
8. Centralności globalne (mediatory / wąskie gardła)
9. Confounding / backdoor: wspólne przyczyny lek → endpoint
10. Gęstość lokalna wokół interakcji i burden
11. Ranking strukturalny leków vs zaprojektowany potencjał ADR
12. Checklist gotowości do modelowania

**Uwaga:** to jest syntetyczna prawda symulacyjna, nie dane kliniczne.


## 0. Setup


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)


def find_dataset_dir() -> Path:
    """Locate 2 v3. Data/dataset_v3 regardless of notebook working directory."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "2 v3. Data" / "dataset_v3"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Nie znaleziono folderu 'dataset_v3'.")


DATA = find_dataset_dir()
OUT = Path(".").resolve() / "diagnostics_outputs"
if not (Path.cwd() / "diagnostics_outputs").exists():
    # if cwd is repo root, write next to this notebook
    nb_dir = Path("3 v3. Graph exploration and diagnostics")
    if nb_dir.exists():
        OUT = nb_dir.resolve() / "diagnostics_outputs"
OUT.mkdir(parents=True, exist_ok=True)

ENDPOINTS = [
    "AKI", "DILI", "Depression", "Falls", "Delirium", "GI_bleeding",
    "Hyponatremia", "Hyperkalemia", "QT_arrhythmia", "Hospitalization",
    "Serotonin_syndrome", "Rhabdomyolysis", "Lactic_acidosis",
]

nodes = pd.read_csv(DATA / "synthetic_pharmacotherapy_v3_nodes.csv")
edges = pd.read_csv(DATA / "synthetic_pharmacotherapy_v3_edges_audited.csv")

G = nx.from_pandas_edgelist(
    edges, "source", "target", edge_attr=True, create_using=nx.DiGraph
)
G.add_nodes_from(nodes["node"])

node_type = nodes.set_index("node")["node_type"].to_dict()
layer = nodes.set_index("node")["layer"].fillna("unknown").astype(str).to_dict()

print("Dane:", DATA)
print("nodes:", G.number_of_nodes())
print("edges:", G.number_of_edges())
print("is_dag:", nx.is_directed_acyclic_graph(G))
print("output dir:", OUT)


## 1. Warstwy, typy węzłów i integralność DAG

Rozkład `node_type` / `layer` pokazuje przepływ:
kontekst pacjenta → leki → mechanizmy → stany ADR → endpointy.

Integralność DAG (acykliczność, spójność, brak self-loopów) potwierdza,
że graf nadaje się do message passing w GNN.


In [ ]:
summary = {
    "n_nodes": G.number_of_nodes(),
    "n_edges": G.number_of_edges(),
    "is_dag": nx.is_directed_acyclic_graph(G),
    "density": round(nx.density(G), 4),
    "weakly_connected_components": nx.number_weakly_connected_components(G),
    "self_loops": nx.number_of_selfloops(G),
    "duplicate_edges": int(edges[["source", "target"]].duplicated().sum()),
    "mean_in_degree": round(np.mean([d for _, d in G.in_degree()]), 3),
    "mean_out_degree": round(np.mean([d for _, d in G.out_degree()]), 3),
}
display(pd.Series(summary, name="value").to_frame())

overview = (
    nodes.groupby(["layer", "node_type"], dropna=False)
    .size()
    .reset_index(name="n_nodes")
    .sort_values(["layer", "node_type"])
)
display(overview)

type_counts = nodes["node_type"].value_counts().rename_axis("node_type").reset_index(name="n")
layer_counts = nodes["layer"].value_counts().rename_axis("layer").reset_index(name="n").sort_values("layer")
display(type_counts)
display(layer_counts)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].barh(type_counts["node_type"], type_counts["n"], color="#3498db")
axes[0].set_title("Liczba węzłów wg node_type")
axes[0].set_xlabel("liczba węzłów")
axes[1].bar(layer_counts["layer"].astype(str), layer_counts["n"], color="#9b59b6")
axes[1].set_title("Liczba węzłów wg warstwy")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

overview.to_csv(OUT / "node_layer_type_overview.csv", index=False)
pd.Series(summary).to_json(OUT / "dag_integrity_summary.json", force_ascii=False, indent=2)


## 2. Stopnie węzłów, huby, źródła i ujścia

- **in-degree** = ile przyczyn wchodzi do węzła
- **out-degree** = na ile węzłów dany węzeł wpływa
- **źródła** (`in=0, out>0`), **ujścia** (`in>0, out=0`), **izolaty** (`in=out=0`)


In [ ]:
degree_rows = []
for node in G.nodes:
    indeg = G.in_degree(node)
    outdeg = G.out_degree(node)
    if indeg == 0 and outdeg == 0:
        role = "isolate"
    elif indeg == 0 and outdeg > 0:
        role = "source"
    elif indeg > 0 and outdeg == 0:
        role = "sink"
    else:
        role = "intermediate"
    degree_rows.append({
        "node": node,
        "node_type": node_type.get(node, "unknown"),
        "layer": layer.get(node, "unknown"),
        "in_degree": indeg,
        "out_degree": outdeg,
        "total_degree": indeg + outdeg,
        "role": role,
        "is_endpoint": node in ENDPOINTS,
    })

deg = pd.DataFrame(degree_rows).sort_values(
    ["total_degree", "out_degree", "in_degree"], ascending=False
)
display(deg["role"].value_counts().rename_axis("role").reset_index(name="n"))

print("Top huby (total_degree):")
display(deg.head(20))

print("Źródła:")
display(deg[deg.role == "source"][["node", "node_type", "out_degree"]])

print("Ujścia:")
display(deg[deg.role == "sink"][["node", "node_type", "in_degree"]])

print("Izolaty:")
display(deg[deg.role == "isolate"][["node", "node_type"]])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].hist(deg["in_degree"], bins=range(0, int(deg["in_degree"].max()) + 2))
axes[0].set_title("In-degree")
axes[1].hist(deg["out_degree"], bins=range(0, int(deg["out_degree"].max()) + 2))
axes[1].set_title("Out-degree")
axes[2].hist(deg["total_degree"], bins=range(0, int(deg["total_degree"].max()) + 2))
axes[2].set_title("Total degree")
for ax in axes:
    ax.set_xlabel("stopień")
    ax.set_ylabel("liczba węzłów")
plt.tight_layout()
plt.show()

print("Średni stopień wg typu węzła:")
display(
    deg.groupby("node_type")[["in_degree", "out_degree", "total_degree"]]
    .mean()
    .round(2)
    .sort_values("total_degree", ascending=False)
)

deg.to_csv(OUT / "node_degree_table.csv", index=False)


## 3. Głębokość topologiczna (warstwy przyczynowe)

Każdemu węzłowi przypisujemy „generację” = najdłuższą odległość od dowolnego źródła.
Pokazuje to, jak głęboki jest łańcuch przyczynowy prowadzący do skutku
i jak głęboki powinien być GNN, żeby objąć długie ścieżki.


In [ ]:
depth = {}
for n in nx.topological_sort(G):
    preds = list(G.predecessors(n))
    depth[n] = 0 if not preds else 1 + max(depth[p] for p in preds)

depth_df = pd.DataFrame({"node": list(depth), "depth": list(depth.values())})
depth_df["node_type"] = depth_df["node"].map(node_type)

print(f"Maksymalna głębokość (najdłuższy łańcuch przyczynowy): {depth_df['depth'].max()} krawędzi")
gen_counts = depth_df["depth"].value_counts().sort_index()

plt.figure(figsize=(9, 4))
plt.bar(gen_counts.index, gen_counts.values, color="#16a085", edgecolor="white")
plt.title("Liczba węzłów na każdej głębokości topologicznej")
plt.xlabel("głębokość (generacja)")
plt.ylabel("liczba węzłów")
plt.show()

print("Najgłębsze węzły:")
display(depth_df.sort_values("depth", ascending=False).head(15))
depth_df.to_csv(OUT / "topological_depth.csv", index=False)


## 4. Typy krawędzi i ścieżki kliniczne

Sprawdzamy, jakie rodzaje relacji dominują w grafie. To pomaga zobaczyć, czy graf ma:

- dużo mechanizmów lekowych,
- dużo interakcji,
- bezpośrednie przejścia do endpointów,
- relacje obserwacyjne.

In [ ]:
edge_type_counts = (
    edges["edge_type"].value_counts().rename_axis("edge_type").reset_index(name="n")
)
pathway_counts = (
    edges["clinical_pathway"].fillna("unknown")
    .value_counts()
    .rename_axis("clinical_pathway")
    .reset_index(name="n")
)
display(edge_type_counts)
display(pathway_counts.head(20))

edge_type_counts.to_csv(OUT / "edge_type_counts.csv", index=False)
pathway_counts.to_csv(OUT / "clinical_pathway_counts.csv", index=False)

## 5. Ścieżki do endpointów: najkrótsze, najdłuższe, liczba dróg

Dla każdego endpointu liczymy:

- liczbę przodków,
- liczbę rodziców bezpośrednich,
- liczbę prostych ścieżek od źródeł (roots) do endpointu,
- najkrótszą i najdłuższą długość ścieżki (w liczbie krawędzi),
- przykładową najkrótszą i najdłuższą ścieżkę.

To pokazuje, czy endpoint jest:

- blisko przyczyn bezpośrednich,
- głęboko osadzony w długich łańcuchach mechanizmów,
- osiągalny na wiele niezależnych sposobów (equifinality).

In [ ]:
def path_stats(target: str) -> dict:
    if target not in G:
        raise KeyError(target)
    ancestors = nx.ancestors(G, target)
    H = G.subgraph(ancestors | {target}).copy()
    topo = list(nx.topological_sort(H))

    shortest = {n: float("inf") for n in H}
    longest = {n: float("-inf") for n in H}
    n_paths = {n: 0 for n in H}
    shortest[target] = 0
    longest[target] = 0
    n_paths[target] = 1

    for node in reversed(topo):
        if node == target:
            continue
        valid = [s for s in H.successors(node) if n_paths[s] > 0]
        if not valid:
            continue
        shortest[node] = min(shortest[s] for s in valid) + 1
        longest[node] = max(longest[s] for s in valid) + 1
        n_paths[node] = sum(n_paths[s] for s in valid)

    roots = [n for n in ancestors if H.in_degree(n) == 0 and n_paths[n] > 0]
    parents = sorted(G.predecessors(target))

    def example(prefer: str):
        if not roots:
            return None
        if prefer == "short":
            root = min(roots, key=lambda r: shortest[r])
            path = [root]
            cur = root
            while cur != target:
                nxt = [
                    s for s in H.successors(cur)
                    if n_paths[s] > 0 and shortest[s] == shortest[cur] - 1
                ] or [s for s in H.successors(cur) if n_paths[s] > 0]
                cur = nxt[0]
                path.append(cur)
            return path
        root = max(roots, key=lambda r: longest[r])
        path = [root]
        cur = root
        while cur != target:
            nxt = [
                s for s in H.successors(cur)
                if n_paths[s] > 0 and longest[s] == longest[cur] - 1
            ] or [s for s in H.successors(cur) if n_paths[s] > 0]
            cur = max(nxt, key=lambda s: longest[s])
            path.append(cur)
        return path

    return {
        "endpoint": target,
        "n_ancestors": len(ancestors),
        "n_parents": len(parents),
        "parents": ", ".join(parents),
        "n_roots": len(roots),
        "n_paths_from_roots": int(sum(n_paths[r] for r in roots)),
        "shortest_len": int(min(shortest[r] for r in roots)) if roots else None,
        "longest_len": int(max(longest[r] for r in roots)) if roots else None,
        "example_shortest": " → ".join(example("short") or []),
        "example_longest": " → ".join(example("long") or []),
    }

path_table = pd.DataFrame([path_stats(ep) for ep in ENDPOINTS if ep in G])
path_table = path_table.sort_values(["n_paths_from_roots", "longest_len"], ascending=False)
display(path_table)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(path_table))
ax.bar(x - 0.2, path_table["shortest_len"], width=0.4, label="shortest")
ax.bar(x + 0.2, path_table["longest_len"], width=0.4, label="longest")
ax.set_xticks(x)
ax.set_xticklabels(path_table["endpoint"], rotation=40, ha="right")
ax.set_ylabel("długość ścieżki (krawędzie)")
ax.set_title("Najkrótsze vs najdłuższe ścieżki do endpointów")
ax.legend()
plt.tight_layout()
plt.show()

path_table.to_csv(OUT / "endpoint_path_diagnostics.csv", index=False)

global_longest = nx.dag_longest_path(G)
print(f"\nGlobalnie najdłuższa ścieżka w DAG ({len(global_longest) - 1} krawędzi):")
print(" -> ".join(global_longest))


### 5.1. Szczegółowa eksploracja jednego endpointu

Zmień `FOCUS_ENDPOINT`, aby przejść przez inny punkt końcowy.

In [ ]:
FOCUS_ENDPOINT = "DILI"  # np. AKI, Falls, Hospitalization, Lactic_acidosis

focus = path_stats(FOCUS_ENDPOINT)
print(pd.Series(focus).to_string())

ancestors = sorted(nx.ancestors(G, FOCUS_ENDPOINT))
ancestor_types = Counter(node_type.get(a, "unknown") for a in ancestors)
print("\nTypy przodków:")
print(pd.Series(ancestor_types).sort_values(ascending=False).to_string())

parents = sorted(G.predecessors(FOCUS_ENDPOINT))
print("\nBezpośredni rodzice:")
for p in parents:
    print(f"- {p} [{node_type.get(p)}]")

## 6. Motywy przyczynowe: mediacja, fork, collider

Dla trzech węzłów `A`, `B`, `C` rozróżniamy:

| Motyw | Struktura | Znaczenie |
|---|---|---|
| **Mediacja (łańcuch)** | `A → B → C` | `B` pośredniczy między `A` a `C` |
| **Fork (wspólna przyczyna)** | `A ← B → C` | `B` powoduje zarówno `A`, jak i `C` |
| **Collider** | `A → B ← C` | `B` jest wspólnym skutkiem `A` i `C` |

Dlaczego to ważne przed modelowaniem?

- mediacja pokazuje ścieżki mechanistyczne,
- fork pokazuje confounding,
- collider pokazuje, gdzie nie należy warunkować (selection / collider bias).

In [ ]:
def motifs_for_center(b: str) -> dict[str, list[tuple[str, str, str]]]:
    parents = list(G.predecessors(b))
    children = list(G.successors(b))
    mediation = [(a, b, c) for a in parents for c in children]
    fork = [(a, b, c) for a, c in combinations(children, 2)]
    collider = [(a, b, c) for a, c in combinations(parents, 2)]
    return {
        "mediation": mediation,
        "fork": fork,
        "collider": collider,
    }

motif_rows = []
all_mediation, all_fork, all_collider = [], [], []

for node in G.nodes:
    m = motifs_for_center(node)
    motif_rows.append({
        "node": node,
        "node_type": node_type.get(node, "unknown"),
        "n_mediation_as_middle": len(m["mediation"]),
        "n_fork_as_common_cause": len(m["fork"]),
        "n_collider_as_common_effect": len(m["collider"]),
    })
    all_mediation.extend(m["mediation"])
    all_fork.extend(m["fork"])
    all_collider.extend(m["collider"])

motif_summary = pd.DataFrame(motif_rows)
print("Łączna liczba motywów lokalnych:")
print({
    "mediation_triplets": len(all_mediation),
    "fork_triplets": len(all_fork),
    "collider_triplets": len(all_collider),
})

print("\nNajczęstsze mediatory (środkowe węzły łańcucha):")
display(
    motif_summary.sort_values("n_mediation_as_middle", ascending=False)
    [["node", "node_type", "n_mediation_as_middle"]].head(15)
)

print("Najczęstsze wspólne przyczyny (fork):")
display(
    motif_summary.sort_values("n_fork_as_common_cause", ascending=False)
    [["node", "node_type", "n_fork_as_common_cause"]].head(15)
)

print("Najczęstsze collidery:")
display(
    motif_summary.sort_values("n_collider_as_common_effect", ascending=False)
    [["node", "node_type", "n_collider_as_common_effect"]].head(15)
)

motif_summary.to_csv(OUT / "motif_counts_by_node.csv", index=False)

### 6.1. Przykłady motywów wokół wybranego węzła

In [ ]:
FOCUS_MOTIF_NODE = "polypharmacy_burden"  # np. ckd, nsaid, cumulative_adr_burden

m = motifs_for_center(FOCUS_MOTIF_NODE)
print(f"Węzeł: {FOCUS_MOTIF_NODE} [{node_type.get(FOCUS_MOTIF_NODE)}]")
print(f"Mediacje (A → {FOCUS_MOTIF_NODE} → C): {len(m['mediation'])}")
for a, b, c in m["mediation"][:10]:
    print(f"  {a} → {b} → {c}")

print(f"\nForki ({FOCUS_MOTIF_NODE} → A oraz {FOCUS_MOTIF_NODE} → C): {len(m['fork'])}")
for a, b, c in m["fork"][:10]:
    print(f"  {a} ← {b} → {c}")

print(f"\nCollidery (A → {FOCUS_MOTIF_NODE} ← C): {len(m['collider'])}")
for a, b, c in m["collider"][:10]:
    print(f"  {a} → {b} ← {c}")

## 7. Przodkowie i potomkowie endpointów

Zbiór przodków endpointu to wszystkie węzły, które mogą na niego wpłynąć.
Rozmiar tego zbioru mówi, jak „szeroki” jest mechanizm prowadzący do zdarzenia.


In [ ]:
FOCUS = "DILI"  # ten sam fokus co w eksploracji ścieżek, można zmienić

anc_desc = pd.DataFrame({
    "endpoint": ENDPOINTS,
    "n_ancestors": [len(nx.ancestors(G, ep)) if ep in G else 0 for ep in ENDPOINTS],
    "n_descendants": [len(nx.descendants(G, ep)) if ep in G else 0 for ep in ENDPOINTS],
}).sort_values("n_ancestors", ascending=False).reset_index(drop=True)
display(anc_desc)

if FOCUS in G:
    focus_anc = nx.ancestors(G, FOCUS)
    by_type = pd.Series([node_type.get(a, "unknown") for a in focus_anc]).value_counts()
    print(f"\nPrzodkowie '{FOCUS}' wg node_type:")
    display(by_type.to_frame("n"))

anc_desc.to_csv(OUT / "endpoint_ancestors_descendants.csv", index=False)


## 8. Centralności globalne — najważniejsze mediatory

- **Betweenness** = jak często węzeł leży na najkrótszych ścieżkach (wąskie gardła).
- **PageRank** = ogólna ważność w przepływie sygnału (uwaga: źródła/leki bywają niskie).

Te centralności liczymy raz dla całego grafu; ranking leków w sekcji 11 z nich korzysta.


In [ ]:
betweenness = nx.betweenness_centrality(G, normalized=True)
pagerank = nx.pagerank(G, alpha=0.85)

central = pd.DataFrame({
    "node": list(G.nodes()),
    "betweenness": [betweenness[n] for n in G.nodes()],
    "pagerank": [pagerank[n] for n in G.nodes()],
})
central["node_type"] = central["node"].map(node_type)

print("TOP 12 wg betweenness (kluczowe mediatory / wąskie gardła):")
display(
    central.sort_values("betweenness", ascending=False)
    .head(12)
    .reset_index(drop=True)
)

print("\nTOP 12 wg PageRank:")
display(
    central.sort_values("pagerank", ascending=False)
    .head(12)
    .reset_index(drop=True)
)

central.to_csv(OUT / "node_centralities.csv", index=False)


## 9. Confounding / backdoor: wspólne przyczyny lek → endpoint

Dla par `lek → endpoint` szukamy **wspólnych przodków** (zmienne wpływające
zarówno na przepisanie leku, jak i na endpoint). To potencjalne confoundery.
`unobserved_severity` jest tu celowym ukrytym confounderem w v3.


In [ ]:
DRUGS = sorted(nodes.loc[nodes["node_type"].eq("drug_exposure"), "node"].tolist())
FOCUS_EPS = ["AKI", "DILI", "Falls", "GI_bleeding", "Hospitalization"]

confound_rows = []
for drug in DRUGS:
    drug_anc = nx.ancestors(G, drug)
    for ep in ENDPOINTS:
        if ep not in G:
            continue
        if drug not in nx.ancestors(G, ep) and not nx.has_path(G, drug, ep):
            continue
        ep_anc = nx.ancestors(G, ep)
        common = sorted(drug_anc & ep_anc)
        if not common and ep not in FOCUS_EPS:
            continue
        confound_rows.append({
            "drug": drug,
            "endpoint": ep,
            "n_common_ancestors": len(common),
            "has_unobserved_severity": "unobserved_severity" in common,
            "has_direct_path_drug_to_ep": nx.has_path(G, drug, ep),
            "common_ancestors_sample": ", ".join(common[:8]),
        })

confound = pd.DataFrame(confound_rows).sort_values(
    ["n_common_ancestors", "has_direct_path_drug_to_ep"], ascending=False
).reset_index(drop=True)

print(f"Par lek→endpoint z co najmniej jednym wspólnym przodkiem: {(confound.n_common_ancestors > 0).sum()}")
display(confound.head(25))

# Przykład jawnego zestawu dla wybranej pary
d, e = "nsaid", "AKI"
if d in DRUGS and e in G:
    common = sorted(nx.ancestors(G, d) & nx.ancestors(G, e))
    print(f"\nWspólni przodkowie dla {d} → {e}:")
    print(common)

confound.to_csv(OUT / "drug_endpoint_common_ancestors.csv", index=False)


## 10. Gęstość lokalna wokół interakcji i obciążenia ADR

Sprawdzamy, czy węzły interakcji i burden są dobrze wpięte w graf:

- ile mają rodziców (składników),
- ile mają dzieci (skutków).

In [ ]:
interesting = deg[
    deg["node"].str.startswith(("ddi_", "drug_disease_"))
    | deg["node"].str.contains("load|burden", regex=True)
].sort_values(["out_degree", "in_degree"], ascending=False)

display(interesting[["node", "node_type", "in_degree", "out_degree", "total_degree"]])
interesting.to_csv(OUT / "interaction_and_burden_degrees.csv", index=False)

## 11. Ranking strukturalny leków vs zaprojektowany potencjał ADR

### Po co to liczymy?

Out-degree sam w sobie jest za prosty: nie rozróżnia siły efektu, długich ścieżek
ani ciężkości skutków. Dlatego liczymy zestaw miar strukturalnych i sprawdzamy,
które najlepiej rankują leki według **zaprojektowanego** potencjału ADR w v3.

### Co liczymy

| Metryka | Sens |
|---|---|
| `out_degree` | liczba bezpośrednich skutków |
| `weighted_out_degree` | suma `effect_size` wychodzących krawędzi |
| `reachable_adr_count` | ile ADR/stanów pośrednich jest osiągalnych |
| `reachable_endpoint_count` | ile endpointów klinicznych jest osiągalnych |
| `reachable_severe_endpoint_count` | ile ciężkich endpointów jest osiągalnych |
| `pagerank` | globalna ważność w grafie |
| `betweenness` | ile ścieżek przechodzi przez lek |
| `katz` | wpływ z uwzględnieniem dłuższych ścieżek |
| `harmonic_closeness` | jak blisko lek jest do reszty grafu |

### Czego nie liczymy tutaj

- liczba receptorów,
- liczba narządów,
- pełne szlaki molekularne.

Te wymagają osobnej warstwy wiedzy biologicznej, której v3 nie ma.

### Jak interpretować korelacje z `clean`?

Porównanie z częstościami ADR w `clean` jest **diagnostyką spójności projektu**,
nie dowodem klinicznym. ADR są generowane z tego samego grafu, więc korelacja
jest częściowo oczekiwana. Pytanie badawcze brzmi:

> Które miary strukturalne najlepiej oddają zaprojektowany potencjał ADR
> i czy późniejszy GNN nauczy się podobnego rankingu?


In [ ]:
ADR_TYPES = {"adr_or_intermediate_state"}
SEVERE_ENDPOINTS = {
    "AKI", "DILI", "GI_bleeding", "QT_arrhythmia", "Hospitalization",
    "Serotonin_syndrome", "Rhabdomyolysis", "Lactic_acidosis",
}

ADR_NODES = nodes.loc[nodes["node_type"].isin(ADR_TYPES), "node"].tolist()
ENDPOINT_NODES = [ep for ep in ENDPOINTS if ep in G]

# Reuse global centralities from section 8; add Katz + harmonic.
katz = nx.katz_centrality_numpy(G, alpha=0.05, beta=1.0)
Gu = G.to_undirected()
harmonic = nx.harmonic_centrality(Gu)

edge_w = edges.set_index(["source", "target"])["effect_size"].to_dict()

rows = []
for drug in DRUGS:
    successors = list(G.successors(drug))
    weighted_out = 0.0
    for tgt in successors:
        val = edge_w.get((drug, tgt), 1.0)
        weighted_out += abs(float(val)) if pd.notna(val) else 1.0

    descendants = nx.descendants(G, drug)
    reachable_adr = [n for n in descendants if node_type.get(n) in ADR_TYPES]
    reachable_ep = [n for n in descendants if n in ENDPOINT_NODES]
    reachable_severe = [n for n in reachable_ep if n in SEVERE_ENDPOINTS]

    rows.append({
        "drug": drug,
        "out_degree": G.out_degree(drug),
        "weighted_out_degree": weighted_out,
        "reachable_adr_count": len(reachable_adr),
        "reachable_endpoint_count": len(reachable_ep),
        "reachable_severe_endpoint_count": len(reachable_severe),
        "pagerank": pagerank[drug],
        "betweenness": betweenness[drug],
        "katz": katz[drug],
        "harmonic_closeness": harmonic[drug],
        "reachable_adrs_sample": ", ".join(sorted(reachable_adr)[:6]),
        "reachable_endpoints_sample": ", ".join(sorted(reachable_ep)[:6]),
    })

drug_struct = pd.DataFrame(rows).sort_values(
    ["reachable_severe_endpoint_count", "weighted_out_degree", "out_degree"],
    ascending=False,
)
print("Top leki według zaprojektowanego potencjału strukturalnego:")
display(drug_struct.head(15))

# Synthetic outcome proxies from clean scenario (consistency check, not clinical proof)
clean_path = DATA / "synthetic_pharmacotherapy_v3_samples_clean.csv"
outcome_cols = ADR_NODES + ENDPOINT_NODES
clean = pd.read_csv(clean_path, usecols=lambda c: c in set(DRUGS + outcome_cols))

proxy_rows = []
for drug in DRUGS:
    exposed = clean[drug].eq(1)
    if exposed.sum() == 0:
        proxy_rows.append({
            "drug": drug,
            "exposure_rate": 0.0,
            "mean_adr_burden_if_exposed": np.nan,
            "any_severe_endpoint_rate_if_exposed": np.nan,
        })
        continue
    sub = clean.loc[exposed]
    adr_present = [c for c in ADR_NODES if c in sub]
    severe_present = [c for c in SEVERE_ENDPOINTS if c in sub]
    mean_adr = sub[adr_present].mean(axis=1).mean() if adr_present else np.nan
    severe_rate = sub[severe_present].max(axis=1).mean() if severe_present else np.nan
    proxy_rows.append({
        "drug": drug,
        "exposure_rate": float(exposed.mean()),
        "mean_adr_burden_if_exposed": float(mean_adr),
        "any_severe_endpoint_rate_if_exposed": float(severe_rate),
    })

proxy = pd.DataFrame(proxy_rows)
ranked = drug_struct.merge(proxy, on="drug", how="left")

metric_cols = [
    "out_degree",
    "weighted_out_degree",
    "reachable_adr_count",
    "reachable_endpoint_count",
    "reachable_severe_endpoint_count",
    "pagerank",
    "betweenness",
    "katz",
    "harmonic_closeness",
]
target_cols = [
    "mean_adr_burden_if_exposed",
    "any_severe_endpoint_rate_if_exposed",
]

corr_rows = []
for metric in metric_cols:
    for target in target_cols:
        valid = ranked[[metric, target]].dropna()
        if len(valid) < 5:
            rho = np.nan
        else:
            rho = valid[metric].corr(valid[target], method="spearman")
        corr_rows.append({
            "metric": metric,
            "synthetic_target": target,
            "spearman_rho": float(rho) if pd.notna(rho) else np.nan,
            "n_drugs": len(valid),
        })

corr = pd.DataFrame(corr_rows).sort_values("spearman_rho", ascending=False)
print("\nKtóre miary najlepiej rankują leki względem syntetycznych proxy ADR?")
display(corr)

print("\nInterpretacja: wyższy Spearman => metryka lepiej oddaje zaprojektowany potencjał ADR.")
print("Uwaga: PageRank bywa niski dla leków-źródeł (upstream) — to normalne w DAG.")
print("To NIE jest walidacja kliniczna real-world ADR.")

ranked.to_csv(OUT / "drug_structural_ranking.csv", index=False)
corr.to_csv(OUT / "drug_metric_vs_synthetic_adr_correlations.csv", index=False)

best = corr.dropna().iloc[0] if not corr.dropna().empty else None
if best is not None:
    print(
        f"\nNajlepsza miara w tym teście: {best['metric']} "
        f"vs {best['synthetic_target']} "
        f"(rho={best['spearman_rho']:.3f})"
    )


## 12. Checklist gotowości do modelowania

Ta sekcja zbiera diagnozę w checklistę przed GNN / baseline.


In [ ]:
checklist = {
    "is_dag": bool(nx.is_directed_acyclic_graph(G)),
    "n_nodes": G.number_of_nodes(),
    "n_edges": G.number_of_edges(),
    "max_topological_depth": int(depth_df["depth"].max()) if "depth_df" in globals() else None,
    "n_isolates": int((deg.role == "isolate").sum()),
    "isolate_names": deg.loc[deg.role == "isolate", "node"].tolist(),
    "n_sources": int((deg.role == "source").sum()),
    "n_sinks": int((deg.role == "sink").sum()),
    "n_endpoints_present": int(sum(ep in G for ep in ENDPOINTS)),
    "endpoints_with_zero_parents": [
        ep for ep in ENDPOINTS if ep in G and G.in_degree(ep) == 0
    ],
    "max_in_degree_node": deg.sort_values("in_degree", ascending=False).iloc[0]["node"] if not deg.empty else None,
    "max_total_degree": int(deg["total_degree"].max()) if not deg.empty else None,
    "n_mediation_triplets": len(all_mediation) if "all_mediation" in globals() else None,
    "n_fork_triplets": len(all_fork) if "all_fork" in globals() else None,
    "n_collider_triplets": len(all_collider) if "all_collider" in globals() else None,
    "hospitalization_paths": (
        int(path_table.loc[path_table.endpoint.eq("Hospitalization"), "n_paths_from_roots"].iloc[0])
        if "path_table" in globals() and "Hospitalization" in set(path_table.endpoint)
        else None
    ),
    "n_drug_endpoint_pairs_with_common_ancestors": (
        int((confound.n_common_ancestors > 0).sum()) if "confound" in globals() else None
    ),
    "n_drugs_ranked": int(len(drug_struct)) if "drug_struct" in globals() else None,
    "best_structural_metric_vs_synthetic_adr": (
        None if "corr" not in globals() or corr.dropna().empty
        else f"{corr.dropna().iloc[0]['metric']} (rho={corr.dropna().iloc[0]['spearman_rho']:.3f})"
    ),
}

print("CHECKLIST GOTOWOŚCI:")
for k, v in checklist.items():
    print(f"- {k}: {v}")

pd.Series(checklist).to_json(OUT / "modeling_readiness_checklist.json", force_ascii=False, indent=2)
print("\nWyniki zapisano w:", OUT)


## 13. Jak czytać wyniki przed modelowaniem?

### Co jest dobre

- graf jest DAG;
- endpointy mają rodziców;
- istnieją zarówno krótkie, jak i długie ścieżki;
- widać mediacje, forki i collidery;
- rankingi strukturalne leków są zgodne z zaprojektowanym potencjałem ADR.

### Na co uważać

- izolaty nie wnoszą relacji do GNN message passing;
- huby o bardzo wysokim stopniu mogą dominować embeddingi;
- długa głębokość topologiczna wymaga większej głębokości GNN;
- collidery nie powinny być bezrefleksyjnie używane jako covariates;
- PageRank bywa mylący dla leków-źródeł;
- korelacje z `clean` są częściowo tautologiczne (ten sam generator).

### Czego tu nie robimy

- nie liczymy liczby receptorów / narządów;
- nie twierdzimy, że metryki przewidują real-world ADR;
- nie trenujemy jeszcze GNN w tym notebooku.

### Co dalej?

1. Eksploruj HTML: `synthetic_pharmacotherapy_v3_dag.html`
2. Porównaj ścieżki i głębokość dla kilku endpointów
3. Sprawdź ranking leków (sekcja 11)
4. Wybierz scenariusz pacjentów (`clean` / `no_overlap` / ...)
5. Dopiero potem uruchamiaj GNN
